# Sleepbox Compass: bulk Qwen OCR on a Colab GPU

Reads Pokémon Sleep detail screenshots with Qwen3.5-4B on a Colab GPU and writes
the answers in the same format as the local read cache
(`data/private/vlm_cache/<model>-p<prompt version>/<sha256>.json`). Copy the
output folder into the local cache and run `pokesleep-box sync`: every image that
Colab already read is a cache hit, so the local run skips inference. Icon
matching, the SP check, evaluation and rendering still run locally.

**Privacy:** this sends your screenshots to Google (the Colab runtime and Google
Drive). Only use it if you accept that; the local `scan`/`sync` never uploads.

vLLM runs in its **own virtual environment** (`/content/vllm-env`, made with `uv`),
so it brings a matching torch/CUDA build and never conflicts with the packages
preinstalled in Colab. Its full log is kept in `/content/vllm.log`; on failure the
error lines are printed.

Steps:
1. Put `inbox.zip` (the screenshots; optionally more zips listed in `INPUT_ZIPS`) in `MyDrive/sleepbox-colab/` on Google Drive.
2. Runtime: a GPU (A100 or L4 recommended).
3. Run all cells. Results go to `MyDrive/sleepbox-colab/<cache folder>/`.
   Images already present there are skipped, so re-running only reads new ones.

In [ ]:
# --- Settings -------------------------------------------------------------
DRIVE_DIR = "/content/drive/MyDrive/sleepbox-colab"   # input zips and output cache
INPUT_ZIPS = ["inbox.zip"]          # zips of screenshots inside DRIVE_DIR (missing ones are skipped)
HF_MODEL = "Qwen/Qwen3.5-4B"        # weights used on the GPU (bf16; MLX 4-bit cannot run on CUDA)
# The local cache folder the results must land in. It is named after the local
# model id so `pokesleep-box sync` (default model) picks the results up.
LOCAL_MODEL = "mlx-community/Qwen3.5-4B-4bit"   # the local default model
REPO = "git+https://github.com/YoheiOhto/sleepbox-compass@main"  # for the exact prompt
VLLM_SPEC = "vllm"                  # pin a version here (e.g. "vllm==X.Y.Z") if the latest fails
# Read from the end of the sorted list so a local `sync` (which reads from the
# start into the same Drive folder) and Colab meet in the middle.
REVERSE = True

In [ ]:
!nvidia-smi
!python --version

In [ ]:
# The project itself (prompt and parser) goes into the Colab kernel; vLLM goes
# into an isolated environment so Colab's preinstalled packages stay untouched.
!pip -q install --no-deps {REPO}
!pip -q install uv
!uv venv -q --python 3.12 /content/vllm-env
!uv pip install -q --python /content/vllm-env/bin/python "{VLLM_SPEC}" huggingface_hub pillow
!uv pip install -q --python /content/vllm-env/bin/python --no-deps {REPO}
!/content/vllm-env/bin/python -c "import vllm, torch; print('vllm', vllm.__version__, '| torch', torch.__version__, '| cuda', torch.version.cuda, torch.cuda.is_available())"

In [ ]:
import hashlib, json, os, re, zipfile
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

from pokesleep_box.vlm import PROMPT, PROMPT_VERSION, parse_response

slug = re.sub(r"[^A-Za-z0-9._-]", "_", LOCAL_MODEL)
out_dir = Path(DRIVE_DIR) / f"{slug}-p{PROMPT_VERSION}"
out_dir.mkdir(parents=True, exist_ok=True)

work = Path("/content/inbox")
work.mkdir(exist_ok=True)
for name in INPUT_ZIPS:
    source = Path(DRIVE_DIR) / name
    if not source.exists():
        print("skip (not found):", source)
        continue
    with zipfile.ZipFile(source) as z:
        z.extractall(work)

suffixes = {".png", ".jpg", ".jpeg", ".webp"}
images = sorted(p for p in work.rglob("*") if p.suffix.lower() in suffixes
                and not p.name.startswith("._"))
todo = []
for p in (reversed(images) if REVERSE else images):
    digest = hashlib.sha256(p.read_bytes()).hexdigest()
    if not (out_dir / f"{digest}.json").exists():
        todo.append((p, digest))
print(f"images: {len(images)}, already read: {len(images) - len(todo)}, to read: {len(todo)}")
print("output:", out_dir)
Path("/content/todo.json").write_text(json.dumps(
    {"model": HF_MODEL, "prompt": PROMPT, "out_dir": str(out_dir),
     "todo": [[str(p), d] for p, d in todo]}, ensure_ascii=False))

In [ ]:
# Batched inference with vLLM inside the isolated environment. Same prompt,
# greedy decoding and no thinking, like the local reader (QwenReader).
SCRIPT = r"""
import base64, json, sys, time
from pathlib import Path
from huggingface_hub import snapshot_download
from vllm import LLM, SamplingParams

job = json.loads(Path("/content/todo.json").read_text())
out_dir = Path(job["out_dir"])
snapshot_download(job["model"])   # download first so engine start-up does not time out
llm = LLM(model=job["model"], max_model_len=8192, limit_mm_per_prompt={"image": 1},
          gpu_memory_utilization=0.85, enforce_eager=True)
params = SamplingParams(temperature=0.0, max_tokens=500)

def conversation(path):
    mime = "image/png" if path.suffix.lower() == ".png" else "image/jpeg"
    data = base64.b64encode(path.read_bytes()).decode()
    return [{"role": "user", "content": [
        {"type": "image_url", "image_url": {"url": f"data:{mime};base64,{data}"}},
        {"type": "text", "text": job["prompt"]}]}]

todo = [(Path(p), d) for p, d in job["todo"]]
BATCH, start, done = 32, time.time(), 0
for i in range(0, len(todo), BATCH):
    # Skip images another engine (e.g. the local sync) finished meanwhile.
    chunk = [(p, d) for p, d in todo[i:i + BATCH] if not (out_dir / f"{d}.json").exists()]
    if chunk:
        results = llm.chat([conversation(p) for p, _ in chunk], params,
                           chat_template_kwargs={"enable_thinking": False}, use_tqdm=False)
        for (p, digest), result in zip(chunk, results):
            (out_dir / f"{digest}.json").write_text(
                json.dumps({"text": result.outputs[0].text}, ensure_ascii=False))
        done += len(chunk)
    print(f"PROGRESS {min(i + BATCH, len(todo))}/{len(todo)}  "
          f"{(time.time() - start) / max(done, 1):.2f} s/image", flush=True)
"""
Path("/content/run_vllm.py").write_text(SCRIPT)

import subprocess
with open("/content/vllm.log", "w") as log:
    proc = subprocess.Popen(["/content/vllm-env/bin/python", "/content/run_vllm.py"],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        log.write(line)
        if line.startswith("PROGRESS"):
            print(line, end="")
    proc.wait()
if proc.returncode:
    lines = Path("/content/vllm.log").read_text().splitlines()
    errors = [l for l in lines if re.search(r"ERROR|Error|error:|CUDA|out of memory|Traceback", l)]
    print("vLLM failed. Error lines (full log: /content/vllm.log):")
    print("\n".join(errors[-40:] or lines[-60:]))

In [ ]:
# Sanity check: how many answers parse into a detail-screen read.
ok = other = bad = 0
for f in out_dir.glob("*.json"):
    screen = parse_response(json.loads(f.read_text())["text"]).get("screen")
    ok += screen == "detail"; other += screen == "other"; bad += screen == "unreadable"
print(f"detail: {ok}, other screens: {other}, unreadable: {bad}")

### If vLLM still fails

Run the cell below instead (Hugging Face transformers in the Colab kernel, one
image at a time; slower but needs no extra installation).

In [ ]:
import time, torch
from PIL import Image
from transformers import AutoModelForImageTextToText, AutoProcessor

processor = AutoProcessor.from_pretrained(HF_MODEL)
model = AutoModelForImageTextToText.from_pretrained(HF_MODEL, torch_dtype=torch.bfloat16,
                                                    device_map="cuda")
start = time.time()
for n, (p, digest) in enumerate(todo, 1):
    if (out_dir / f"{digest}.json").exists():
        continue  # read meanwhile by another engine
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": PROMPT}]}]
    prompt = processor.apply_chat_template(messages, add_generation_prompt=True,
                                           enable_thinking=False)
    inputs = processor(text=[prompt], images=[Image.open(p).convert("RGB")],
                       return_tensors="pt").to("cuda")
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=500, do_sample=False)
    text = processor.batch_decode(output[:, inputs["input_ids"].shape[1]:],
                                  skip_special_tokens=True)[0]
    (out_dir / f"{digest}.json").write_text(json.dumps({"text": text}, ensure_ascii=False))
    print(f"{n}/{len(todo)}  {(time.time() - start) / n:.2f} s/image")